In [4]:
import pandas as pd

In [5]:
from pathlib import Path

BASE_DIR = Path("../data")

GLOBAL_WEEKLY_FILE = BASE_DIR / "raw" / "global_weekly.xlsx"
GLOBAL_ALLTIME_FILE = BASE_DIR / "raw" / "global_alltime.xlsx"

OUTPUT_FILE = BASE_DIR / "processed" / "netflix.csv"

# Ändra sökvägen om filerna ligger i en annan map.
# Det är viktigt att ni väljer filerna i den ordningen som finns nedan. 

GLOBAL_WEEKLY_FILE = "raw_data/2026-02-08_global_weekly.xlsx" # Här skall den global_weekly.xlsx ligga
GLOBAL_ALLTIME_FILE = "raw_data/2026-02-08_global_alltime.xlsx" # Här skall global_alltime.xlsx ligga
OUTPUT_FILE = "processed_data/netflix.csv"

#### 1. läser in filen  "2026-02-08_global_weekly.xlsx" till en DataFrame 

In [6]:
df_global_weekly = pd.read_excel(GLOBAL_WEEKLY_FILE)

c:\Users\adelo\de25\Netflix_Analytics_DE_UX\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


#### 2. läser in filen "2026-02-08_global_alltime.xlsx" till en DataFrame 

In [7]:
df_global_alltime = pd.read_excel(GLOBAL_ALLTIME_FILE)

c:\Users\adelo\de25\Netflix_Analytics_DE_UX\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


#### 3. Konverterar "week" kolumnen i båda DataFrames till formatet datetime

In [8]:
df_global_alltime["week"] = pd.to_datetime(df_global_alltime["week"])
df_global_weekly["week"] = pd.to_datetime(df_global_weekly["week"])

#### 4. Undersöker unika kategorier i filen "2026-02-08_global_weekly.xlsx" 

In [9]:
df_global_weekly["category"].unique()

array(['Films', 'TV'], dtype=object)

#### 5. Undersöker unika kategorier i filen "2026-02-08_global_alltime.xlsx" 

In [10]:
df_global_alltime["category"].unique()

array(['Films (English)', 'Films (Non-English)', 'TV (English)',
       'TV (Non-English)'], dtype=object)

#### 6. Sätter prefix på kolumnerna i global_alltime (2026-02-08_global_alltime.xlsx)

In [11]:

df_global_alltime = df_global_alltime.rename(columns={'category': 'global_category', 'weekly_views': 'global_weekly_views', 'weekly_hours_viewed': 'global_weekly_hours_viewed',
                                  'weekly_rank': 'global_weekly_rank','cumulative_weeks_in_top_10':'global_cumulative_weeks_in_top_10'})

df_global_alltime

,week,global_category,global_weekly_rank,show_title,season_title,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,2026-03-15,Films (English),1,War Machine,NaN,80600000,1.8167,44400000.0,2
1,2026-03-15,Films (English),2,Louis Theroux: Inside the Manosphere,NaN,9600000,1.5167,6300000.0,1
2,2026-03-15,Films (English),3,Shark Tale,NaN,8100000,1.5000,5400000.0,2
3,2026-03-15,Films (English),4,KPop Demon Hunters,NaN,8800000,1.6667,5300000.0,39
4,2026-03-15,Films (English),5,Double Jeopardy,NaN,6200000,1.7667,3500000.0,1
...,...,...,...,...,...,...,...,...,...
9835,2021-07-04,TV (Non-English),6,Elite,Elite: Season 1,10530000,NaN,NaN,1
9836,2021-07-04,TV (Non-English),7,Elite,Elite: Season 3,10200000,NaN,NaN,1
9837,2021-07-04,TV (Non-English),8,Elite,Elite: Season 2,10140000,NaN,NaN,1
9838,2021-07-04,TV (Non-English),9,Katla,Katla: Season 1,9190000,NaN,NaN,1


#### 7. Sätter prefix på kolumnerna i global_weekly (2026-02-08_global_weekly.xlsx)

In [12]:
df_global_weekly = df_global_weekly.rename(columns={'category': 'country_category', 'weekly_rank': 'country_weekly_rank',
                                                    'cumulative_weeks_in_top_10':'country_cumulative_weeks_in_top_10' })

df_global_weekly

,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10
0,Argentina,AR,2026-03-15,Films,1,War Machine,NaN,2
1,Argentina,AR,2026-03-15,Films,2,Strangers in the Park,NaN,2
2,Argentina,AR,2026-03-15,Films,3,Joker: Folie à Deux,NaN,1
3,Argentina,AR,2026-03-15,Films,4,Trolls Band Together,NaN,2
4,Argentina,AR,2026-03-15,Films,5,Double Jeopardy,NaN,1
...,...,...,...,...,...,...,...,...
458255,Vietnam,VN,2021-07-04,TV,6,Reply 1988,Reply 1988: Season 1,1
458256,Vietnam,VN,2021-07-04,TV,7,"Nevertheless,","Nevertheless,: Limited Series",1
458257,Vietnam,VN,2021-07-04,TV,8,Too Hot to Handle,Too Hot to Handle: Season 2,1
458258,Vietnam,VN,2021-07-04,TV,9,Record of Ragnarok,Record of Ragnarok: Season 1,1


#### 8. Konverterar NaN till "N/A"

In [13]:
df_global_weekly["season_title"] = df_global_weekly["season_title"].fillna("N/A")
df_global_alltime["season_title"] = df_global_alltime["season_title"].fillna("N/A")

#### 9. Gör om kolumnen "runtime" till minuter för att få mer detaljerad data

In [14]:
df_global_alltime["runtime"] = df_global_alltime["runtime"] * 60

#### 10. Tittar efter dubletter

In [15]:
print(f"Antal dubletter {len(df_global_alltime) - len(df_global_alltime.drop_duplicates())}")

Antal dubletter 0


In [16]:
print(f"Antal dubletter {len(df_global_weekly) - len(df_global_weekly.drop_duplicates())}")

Antal dubletter 0


#### 11. Slår ihop båda DataFrames

In [17]:
Netflix_df = df_global_weekly.merge(df_global_alltime, on=['week','show_title','season_title'], how='left')

#### 12.Skriver ut DataFramen till en csv-fil 

In [18]:
Netflix_df.to_csv(OUTPUT_FILE)

In [19]:
Netflix_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 458338 entries, 0 to 458337
Data columns (total 14 columns):
 #   Column                              Non-Null Count   Dtype         
---  ------                              --------------   -----         
 0   country_name                        458338 non-null  object        
 1   country_iso2                        458338 non-null  object        
 2   week                                458338 non-null  datetime64[ns]
 3   country_category                    458338 non-null  object        
 4   country_weekly_rank                 458338 non-null  int64         
 5   show_title                          458338 non-null  object        
 6   season_title                        458338 non-null  object        
 7   country_cumulative_weeks_in_top_10  458338 non-null  int64         
 8   global_category                     306075 non-null  object        
 9   global_weekly_rank                  306075 non-null  float64       
 10  global_w